# Kodr — build your game-generation AI

This notebook trains **Kodr**: a standalone, open-source AI that writes game code, systems, and maps for *any* engine (Roblox Luau, Unity C#, Godot GDScript, Unreal C++).

Pipeline (as designed by the owner):
1. **Grab a reference model** (Qwen3-class coder).
2. **Fine-tune** it on the Kodr game-dev dataset with QLoRA.
3. **Merge** the LoRA adapter into the base weights → one standalone model.
4. **Delete the reference model fully** — HF cache, adapter, everything. Only your merged Kodr remains.
5. Push Kodr to Hugging Face, then chat with it in the free playground Space.

In [ ]:
# 1. Install the training stack
!pip -q install torch transformers peft trl datasets accelerate bitsandbytes huggingface_hub safetensors

In [ ]:
# 2. Get the Kodr repo + training kit
!git clone --depth 1 https://github.com/Radin-dev1/kodr.git && echo OK

In [ ]:
# 3. Log in to Hugging Face
from huggingface_hub import notebook_login
import os
notebook_login()  # paste a WRITE token (https://huggingface.co/settings/tokens)

# Pick the final public name of your Kodr model repo
os.environ['KODR_REPO'] = input('Model repo name (default kodr): ') or 'kodr'

In [ ]:
# 4. Auto-size the reference model to the GPU we actually have
import torch, pprint

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB')
else:
    print('WARNING: No GPU detected - a 27B reference is impossible on CPU.\n'
          'Use "Runtime > Change runtime type > T4 GPU" (free) or A100 (paid) and rerun.')

# The owner's preferred reference models:
REFERENCE_27B = 'Qwen/Qwen3.8-27B'                       # flags reference
REFERENCE_KAT = 'Kwaipilot/KAT-Coder-V2.5-Dev'
REFERENCE_TURBO = 'DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF'

def pick_base():
    if torch.cuda.is_available():
        vram = torch.cuda.get_device_properties(0).total_memory / 2**30
        if vram >= 40:  # A100 / L40S class -> the full 27B reference
            return REFERENCE_27B
        if vram >= 24:  # L4 / 4090 -> Qwen3 8B-class coder variant
            return REFERENCE_KAT if False else 'Qwen/Qwen2.5-Coder-7B-Instruct'
    # T4 (16GB) -> the best trainable small coder
    return 'Qwen/Qwen2.5-Coder-3B-Instruct'

BASE = pick_base()
print('Reference model for this run:', BASE)

In [ ]:
# 5. TRAIN. Merges -> deletes the reference fully -> pushes your standalone model.
%cd /content/kodr/model
!python train.py \
  --dataset dataset/kodr-dataset.jsonl \
  --base-model "$BASE" \
  --repo "$KODR_REPO" \
  --epochs 3 \
  --push

In [ ]:
# 6. Output lines you should see:
#     1. training loss descending below ~0.9
#     2. ">> Training done. Merging LoRA into the base weights..."
#     3. ">> Removing the reference model completely..."
#     4. ">> DONE. Your standalone model: https://huggingface.co/<user>/kodr"
print('All good. Open the HF repo, set it public, and the playground Space\n'
      '(huggingface.co/spaces/Radinkazemian/kodr-playground) will serve it.')